<a href="https://colab.research.google.com/github/rahidparvez/tihm-dp-pilot/blob/main/TIHM_DP_pilot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pilot: privacy-preserving data quality checks on TIHM home-monitoring data

A small pilot that replicates, in Python, the Laplace approach of Tomášik et al. (BMC Medical Informatics and Decision Making, 26:49, 2026) on the open TIHM dataset. It is not an extension of their data quality agent.

It looks at three questions: why patient-level contribution bounding matters, how differential privacy noise affects quality metrics in small cohorts, and which continual release method is more accurate for weekly reports.

Data: TIHM (Palermo et al., Scientific Data 10:606, 2023), downloaded directly from Zenodo (record 7622128) in the first cell; no data is stored in this repository. TIHM is licensed under CC-BY-4.0 for non-commercial research use. We acknowledge Surrey and Borders Partnership NHS Foundation Trust.

Author: Md. Rahid Parvez

In [1]:
# Download TIHM directly from Zenodo and unzip it
!wget -q -O TIHM_Dataset.zip "https://zenodo.org/records/7622128/files/TIHM_Dataset.zip?download=1"
!unzip -q -o TIHM_Dataset.zip

import numpy as np, pandas as pd, matplotlib.pyplot as plt

DATA_DIR = "Dataset"   # folder created by the unzip step
EPS = 1.0              # total privacy budget (epsilon) per check
REPS = 2000            # number of repetitions per experiment
rng = np.random.default_rng(42)   # fixed seed for reproducible results

phys = pd.read_csv(f"{DATA_DIR}/Physiology.csv", parse_dates=["date"])
print(phys.shape, phys.patient_id.nunique(), "patients,", phys.date.min().date(), "to", phys.date.max().date())
print(phys.device_type.value_counts())

(17679, 5) 55 patients, 2019-04-01 to 2019-06-30
device_type
Body Temperature            5617
Systolic blood pressure     2305
Diastolic blood pressure    2305
Heart rate                  2305
Body weight                 1874
O/E - muscle mass           1264
Total body water            1264
Skin Temperature             745
Name: count, dtype: int64


In [2]:
hr = phys[phys.device_type=="Heart rate"].sort_values(["patient_id","date"])

# Check A: any gap of more than 7 days between consecutive heart rate readings
max_gap_days = hr.groupby("patient_id").date.apply(
    lambda s: s.diff().dt.total_seconds().max()/86400).dropna()
A = (max_gap_days > 7).astype(int)

# Check B: any body temperature reading outside 34-42 C
bt = phys[phys.device_type=="Body Temperature"]
B = bt.groupby("patient_id").value.apply(lambda v: int(((v<34)|(v>42)).any()))

for name, s in [("A: gap > 7 days", A), ("B: temp outside 34-42 C", B)]:
    print(f"{name}: {s.sum()} of {len(s)} patients = {100*s.mean():.1f}%")

A: gap > 7 days: 16 of 53 patients = 30.2%
B: temp outside 34-42 C: 2 of 55 patients = 3.6%


In [3]:
# Counting gap events instead of patients lets one patient contribute several times
gap_events = hr.groupby("patient_id").date.apply(
    lambda s: int((s.diff().dt.total_seconds()/86400 > 7).sum()))
sens_event = gap_events.max()

print("Event level:   total =", gap_events.sum(), "| sensitivity =", sens_event, "| Laplace scale =", sens_event/EPS)
print("Patient level: count =", A.sum(), "| sensitivity = 1 | Laplace scale =", 1/EPS)
print("Max HR readings by one patient:", hr.groupby("patient_id").size().max())

Event level:   total = 24 | sensitivity = 3 | Laplace scale = 3.0
Patient level: count = 16 | sensitivity = 1 | Laplace scale = 1.0
Max HR readings by one patient: 163


In [4]:
def dp_ratio(c, n, eps, rng):
    # noise on both numerator and denominator; budget split in two (as in the BMC paper)
    c_t = c + rng.laplace(0, 2/eps)
    n_t = n + rng.laplace(0, 2/eps)
    return float(np.clip(c_t / max(n_t, 1), 0, 1))

def cohort_experiment(s, sizes, eps=EPS, reps=REPS):
    vals = s.values; rows = []
    for m in sizes:
        err = []
        for _ in range(reps):
            sub = rng.choice(vals, size=m, replace=False)          # random subsample
            err.append(abs(dp_ratio(sub.sum(), m, eps, rng) - sub.mean()) * 100)
        err = np.array(err)
        rows.append({"cohort": m,
                     "median error (pp)": round(np.median(err), 1),
                     "95th pct error (pp)": round(np.percentile(err, 95), 1)})
    return pd.DataFrame(rows)

resA = cohort_experiment(A, [len(A), 30, 10])
resB = cohort_experiment(B, [len(B), 30, 10])
print("Check A (true rate %.1f%%)" % (100*A.mean())); print(resA.to_string(index=False))
print("\nCheck B (true rate %.1f%%)" % (100*B.mean())); print(resB.to_string(index=False))

Check A (true rate 30.2%)
 cohort  median error (pp)  95th pct error (pp)
     53                2.8                 12.0
     30                5.1                 20.5
     10               14.0                 54.8

Check B (true rate 3.6%)
 cohort  median error (pp)  95th pct error (pp)
     55                2.5                  8.4
     30                3.3                 15.7
     10                5.5                 50.0


In [5]:
start = pd.Timestamp("2019-04-01")
d = hr.copy(); d["prev"] = d.groupby("patient_id").date.shift()
d = d[(d.date - d.prev).dt.total_seconds()/86400 > 7]
first_gap_week = d.groupby("patient_id").date.min().sub(start).dt.days // 7   # each patient counted once, at first gap
T = int((phys.date.max() - start).days // 7) + 1
inc = np.bincount(first_gap_week.values, minlength=T)[:T]
true_cum = np.cumsum(inc)
print("weeks T =", T, "| weekly new:", inc.tolist(), "| final:", true_cum[-1])

def naive_cum(eps):          # noise added to each weekly cumulative count
    return true_cum + rng.laplace(0, T/eps, T)

def sum_inc(eps):            # noise added to weekly new counts, then summed
    return np.cumsum(inc + rng.laplace(0, 1/eps, T))

def binary_tree(eps):        # binary tree mechanism (Chan et al., 2011)
    L = int(np.ceil(np.log2(T))) + 1
    noisy = {}; out = np.zeros(T)
    for t in range(1, T+1):
        total, start_i = 0.0, 0
        for lev in range(L-1, -1, -1):
            size = 2**lev
            if start_i + size <= t:
                key = (start_i, size)
                if key not in noisy:
                    noisy[key] = inc[start_i:start_i+size].sum() + rng.laplace(0, L/eps)
                total += noisy[key]; start_i += size
        out[t-1] = total
    return out

rows = []
for name, f in [("naive cumulative", naive_cum), ("sum of noisy increments", sum_inc), ("binary tree", binary_tree)]:
    errs = np.array([f(EPS) - true_cum for _ in range(REPS)])
    rows.append({"method": name,
                 "RMSE all weeks": round(np.sqrt((errs**2).mean()), 2),
                 "RMSE final week": round(np.sqrt((errs[:, -1]**2).mean()), 2)})
cont = pd.DataFrame(rows)
print(cont.to_string(index=False))

weeks T = 13 | weekly new: [0, 0, 0, 1, 2, 2, 1, 1, 1, 2, 2, 3, 1] | final: 16
                 method  RMSE all weeks  RMSE final week
       naive cumulative           18.25            18.64
sum of noisy increments            3.71             5.07
            binary tree            9.85            12.12


In [6]:
def tree_err(T, eps=EPS):
    L = int(np.ceil(np.log2(T))) + 1; noisy = {}; out = np.zeros(T)
    for t in range(1, T+1):
        total, start_i = 0.0, 0
        for lev in range(L-1, -1, -1):
            size = 2**lev
            if start_i + size <= t:
                key = (start_i, size)
                if key not in noisy: noisy[key] = rng.laplace(0, L/eps)
                total += noisy[key]; start_i += size
        out[t-1] = total
    return np.sqrt((out**2).mean())

rows = []
for Tn, reps in [(13, 300), (64, 300), (512, 100), (4096, 20)]:
    inc_rmse = np.mean([np.sqrt((np.cumsum(rng.laplace(0, 1/EPS, Tn))**2).mean()) for _ in range(reps)])
    tr_rmse = np.mean([tree_err(Tn) for _ in range(reps)])
    rows.append({"releases T": Tn, "increments RMSE": round(inc_rmse, 1), "binary tree RMSE": round(tr_rmse, 1)})
print(pd.DataFrame(rows).to_string(index=False))

 releases T  increments RMSE  binary tree RMSE
         13              3.4               9.3
         64              6.7              16.6
        512             18.6              29.8
       4096             49.8              42.7


In [7]:
a_full, a10 = resA.iloc[0], resA.iloc[-1]
print(f"On TIHM ({len(A)} patients with HR data), a patient-level gap check at eps = 1 had a median "
      f"absolute error of {a_full['median error (pp)']} pp on the full cohort and "
      f"{a10['median error (pp)']} pp on random 10-patient cohorts.")
best = cont.sort_values("RMSE all weeks").iloc[0]
print(f"For {T} weekly releases under a total eps = 1, the lowest RMSE came from: {best['method']} "
      f"({best['RMSE all weeks']} patients).")

On TIHM (53 patients with HR data), a patient-level gap check at eps = 1 had a median absolute error of 2.8 pp on the full cohort and 14.0 pp on random 10-patient cohorts.
For 13 weekly releases under a total eps = 1, the lowest RMSE came from: sum of noisy increments (3.71 patients).
